In [11]:
import sys
import warnings
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
sys.path.insert(0, "../../Utilities/")
import common_functions as cf

warnings.filterwarnings('ignore')

In [12]:
data_in = np.load('initial_inputs.npy')
data_out = np.load('initial_outputs.npy')
#print(data_in)
#print(data_out)

Week-01

In [13]:
X_init = np.array([
[0.89698105, 0.72562797, 0.17540431, 0.70169437],
[0.8893564 , 0.49958786, 0.53926886, 0.50878344],
[0.25094624, 0.03369313, 0.14538002, 0.49493242],
[0.34696206, 0.0062504 , 0.76056361, 0.61302356],
[0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
[0.80130271, 0.50023109, 0.70664456, 0.19510284],
[0.24770826, 0.06044543, 0.04218635, 0.44132425],
[0.74670224, 0.7570915 , 0.36935306, 0.20656628],
[0.40066503, 0.07257425, 0.88676825, 0.24384229],
[0.6260706 , 0.58675126, 0.43880578, 0.77885769],
[0.95713529, 0.59764438, 0.76611385, 0.77620991],
[0.73281243, 0.14524998, 0.47681272, 0.13336573],
[0.65511548, 0.07239183, 0.68715175, 0.08151656],
[0.21973443, 0.83203134, 0.48286416, 0.08256923],
[0.48859419, 0.2119651 , 0.93917791, 0.37619173],
[0.16713049, 0.87655456, 0.21723954, 0.95980098],
[0.21691119, 0.16608583, 0.24137226, 0.77006248],
[0.38748784, 0.80453226, 0.75179548, 0.72382744],
[0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
[0.03782483, 0.66485335, 0.16198218, 0.25392378],
[0.68348638, 0.9027701 , 0.33541983, 0.99948256],
[0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
[0.85965692, 0.91959232, 0.20613873, 0.09779683],
[0.28213837, 0.50598691, 0.53053084, 0.09630162],
[0.32607578, 0.4723669 , 0.453192  , 0.10588734],
[0.94838936, 0.89451301, 0.85163782, 0.55219629],
[0.66495539, 0.04656628, 0.11677747, 0.79371778],
[0.57776561, 0.42877174, 0.42582587, 0.24900741],
[0.73861301, 0.48210263, 0.70936644, 0.50397001],
[0.8548108 , 0.49396462, 0.73530997, 0.80809201]
])

y_init = np.array([
-22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
-15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
-27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
-26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
-29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
-32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284]
)

X_init = data_in
y_init = data_out

y_trans = -y_init
y_best = y_trans.max()

In [14]:
# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_trans)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu = mu[0]
    sigma = sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because we minimize in scipy
# --- Expected Improvement acquisition function ---

# --- Optimize acquisition function ---
bounds = [(0,1), (0,1), (0,1), (0,1)]
best_x = None
best_ei = float('inf')

# Multiple random starts to avoid local maxima
for _ in range(20):
    x0 = np.random.rand(4)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)


Next hyperparameter set to try: [1.         1.         0.91020333 0.73869489]


Week-02

In [15]:
X_init = data_in
y_init = data_out
# --- Transform to maximization ---
y_trans = -y_init
y_best = y_trans.max()


# Add the new observation & Combine with previous data
X_all = np.vstack(
    [
        X_init, 
        [0.531713, 0.402180, 0.000000, 0.275240],
        [1.000000,1.000000,0.910302,0.738550]
    ])

y_all = np.hstack([
    y_init, 
    [-10.311800191097564],
    [-43.88429740682509]
    ])

y_trans = -y_all

y_best = y_trans.max()

print(X_all)
print(y_all)


[[0.89698105 0.72562797 0.17540431 0.70169437]
 [0.8893564  0.49958786 0.53926886 0.50878344]
 [0.25094624 0.03369313 0.14538002 0.49493242]
 [0.34696206 0.0062504  0.76056361 0.61302356]
 [0.12487118 0.12977019 0.38440048 0.2870761 ]
 [0.80130271 0.50023109 0.70664456 0.19510284]
 [0.24770826 0.06044543 0.04218635 0.44132425]
 [0.74670224 0.7570915  0.36935306 0.20656628]
 [0.40066503 0.07257425 0.88676825 0.24384229]
 [0.6260706  0.58675126 0.43880578 0.77885769]
 [0.95713529 0.59764438 0.76611385 0.77620991]
 [0.73281243 0.14524998 0.47681272 0.13336573]
 [0.65511548 0.07239183 0.68715175 0.08151656]
 [0.21973443 0.83203134 0.48286416 0.08256923]
 [0.48859419 0.2119651  0.93917791 0.37619173]
 [0.16713049 0.87655456 0.21723954 0.95980098]
 [0.21691119 0.16608583 0.24137226 0.77006248]
 [0.38748784 0.80453226 0.75179548 0.72382744]
 [0.98562189 0.66693268 0.15678328 0.8565348 ]
 [0.03782483 0.66485335 0.16198218 0.25392378]
 [0.68348638 0.9027701  0.33541983 0.99948256]
 [0.17034731 

In [16]:
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_all)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".","Matern(length_scale=1, nu=2.5)"
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-06
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",0
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",True
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`

In [17]:
# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu = mu[0]
    sigma = sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because we minimize in scipy

In [18]:
# --- Bounds ---
bounds = [(0, 1)] * 4

# --- Optimize acquisition function ---
best_x = None
best_ei = float('inf')

# 1️⃣ Exploitation: many starts near the last best point for fine refinement
for _ in range(10):
    x0 = X_all[0] + np.random.normal(0, 0.02, size=4)  # small noise
    x0 = np.clip(x0, 0, 1)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

# 2️⃣ Exploration: a few random starts to cover the rest of the space
for _ in range(5):
    x0 = np.random.rand(4)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)


Next hyperparameter set to try: [0.99105252 0.9485424  0.01093901 0.90723213]
0.991053-0.948542-0.010939-0.907232


Week-04

In [19]:
X_init = np.array([
[0.89698105, 0.72562797, 0.17540431, 0.70169437],
[0.8893564 , 0.49958786, 0.53926886, 0.50878344],
[0.25094624, 0.03369313, 0.14538002, 0.49493242],
[0.34696206, 0.0062504 , 0.76056361, 0.61302356],
[0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
[0.80130271, 0.50023109, 0.70664456, 0.19510284],
[0.24770826, 0.06044543, 0.04218635, 0.44132425],
[0.74670224, 0.7570915 , 0.36935306, 0.20656628],
[0.40066503, 0.07257425, 0.88676825, 0.24384229],
[0.6260706 , 0.58675126, 0.43880578, 0.77885769],
[0.95713529, 0.59764438, 0.76611385, 0.77620991],
[0.73281243, 0.14524998, 0.47681272, 0.13336573],
[0.65511548, 0.07239183, 0.68715175, 0.08151656],
[0.21973443, 0.83203134, 0.48286416, 0.08256923],
[0.48859419, 0.2119651 , 0.93917791, 0.37619173],
[0.16713049, 0.87655456, 0.21723954, 0.95980098],
[0.21691119, 0.16608583, 0.24137226, 0.77006248],
[0.38748784, 0.80453226, 0.75179548, 0.72382744],
[0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
[0.03782483, 0.66485335, 0.16198218, 0.25392378],
[0.68348638, 0.9027701 , 0.33541983, 0.99948256],
[0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
[0.85965692, 0.91959232, 0.20613873, 0.09779683],
[0.28213837, 0.50598691, 0.53053084, 0.09630162],
[0.32607578, 0.4723669 , 0.453192  , 0.10588734],
[0.94838936, 0.89451301, 0.85163782, 0.55219629],
[0.66495539, 0.04656628, 0.11677747, 0.79371778],
[0.57776561, 0.42877174, 0.42582587, 0.24900741],
[0.73861301, 0.48210263, 0.70936644, 0.50397001],
[0.8548108 , 0.49396462, 0.73530997, 0.80809201]
])

y_init = np.array([
-22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
-15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
-27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
-26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
-29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
-32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284
])

X_all = np.vstack([
X_init, 
[0.531713,0.402180,0.000000,0.275240],
[1.000000,1.000000,0.910302,0.738550],
[0.136128,0.968250,0.991906,0.639381]
])

y_all = np.hstack([
y_init, 
-10.311800191097564,
-43.88429740682509,
-32.802627003504426
])


eps= 1e-20
signs = np.sign(y_all)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_all) + eps)


kernel = Matern(length_scale = 0.1, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_trans)

def acquisition_ei(X, gp, y_best, xi=0.05):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.03
n = 6

# Create 1D ranges
x1 = np.linspace(margin, 1-margin, n)
x2 = np.linspace(margin, 1-margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

# Create 4D meshgrid
X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

#bounds = [(X_all[:,i].min(), X_all[:,i].max()) for i in range(4)]
#num_candidates = 5000
#X_candidates = np.column_stack([
#    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
#])

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"[{next[0]:.6f}, {next[1]:.6f}, {next[2]:.6f}, {next[3]:.6f}]")

print(cf.format_inputdata(next))

[0.030000, 0.030000, 0.970000, 0.970000]
0.030000-0.030000-0.970000-0.970000


Week-05

In [20]:
from sklearn.gaussian_process.kernels import ConstantKernel, WhiteKernel
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [21]:
X_init = np.array([
[0.89698105, 0.72562797, 0.17540431, 0.70169437],
[0.8893564 , 0.49958786, 0.53926886, 0.50878344],
[0.25094624, 0.03369313, 0.14538002, 0.49493242],
[0.34696206, 0.0062504 , 0.76056361, 0.61302356],
[0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
[0.80130271, 0.50023109, 0.70664456, 0.19510284],
[0.24770826, 0.06044543, 0.04218635, 0.44132425],
[0.74670224, 0.7570915 , 0.36935306, 0.20656628],
[0.40066503, 0.07257425, 0.88676825, 0.24384229],
[0.6260706 , 0.58675126, 0.43880578, 0.77885769],
[0.95713529, 0.59764438, 0.76611385, 0.77620991],
[0.73281243, 0.14524998, 0.47681272, 0.13336573],
[0.65511548, 0.07239183, 0.68715175, 0.08151656],
[0.21973443, 0.83203134, 0.48286416, 0.08256923],
[0.48859419, 0.2119651 , 0.93917791, 0.37619173],
[0.16713049, 0.87655456, 0.21723954, 0.95980098],
[0.21691119, 0.16608583, 0.24137226, 0.77006248],
[0.38748784, 0.80453226, 0.75179548, 0.72382744],
[0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
[0.03782483, 0.66485335, 0.16198218, 0.25392378],
[0.68348638, 0.9027701 , 0.33541983, 0.99948256],
[0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
[0.85965692, 0.91959232, 0.20613873, 0.09779683],
[0.28213837, 0.50598691, 0.53053084, 0.09630162],
[0.32607578, 0.4723669 , 0.453192  , 0.10588734],
[0.94838936, 0.89451301, 0.85163782, 0.55219629],
[0.66495539, 0.04656628, 0.11677747, 0.79371778],
[0.57776561, 0.42877174, 0.42582587, 0.24900741],
[0.73861301, 0.48210263, 0.70936644, 0.50397001],
[0.8548108 , 0.49396462, 0.73530997, 0.80809201],
[0.531713,0.402180,0.000000,0.275240], #week-01 new input
[1.000000,1.000000,0.910302,0.738550], #week-02 new input
[0.136128,0.968250,0.991906,0.639381], #week-03 new input
[0.030000, 0.030000, 0.970000, 0.970000] #week-04 new input
])

y_init = np.array([
-22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
-15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
-27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
-26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
-29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
-32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284,
-10.311800191097564, #week-01 new output
-43.88429740682509, #week-02 new output
-32.802627003504426, #week-03 new output
-36.57670166890428 #week-04 new output
])

eps= 1e-20
signs = np.sign(y_init)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_init) + eps)

#kernel = Matern(length_scale = 0.1, nu=2.5)
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.25, length_scale_bounds=(1e-2, 2.0), nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-10, 1e-1))

#gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-8, 
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
    )

gp.fit(X_init, y_trans)

rf = RandomForestRegressor(
    n_estimators=600, 
    random_state=42,
    bootstrap=True,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
    )

rf.fit(X_init, y_trans)

gbm_ens=[]
for k in range(12):
    gbm = GradientBoostingRegressor(
        n_estimators=350, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=100+k,
        subsample=0.8,
        min_samples_leaf=2
        )
    gbm.fit(X_init, y_trans)
    gbm_ens.append(gbm)



def acquisition_ei(X, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.02
n=6
x1 = np.linspace(margin, 1 - margin, n)
x2 = np.linspace(margin, 1 - margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4,
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next_point = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"Next point using ei: [{next_point[0]:.6f}, {next_point[1]:.6f}, {next_point[2]:.6f}, {next_point[3]:.6f}]")

#res_formatted = [f"{r:.6f}" for r in next_point]
#result = "-".join(res_formatted)
print(cf.format_inputdata(next_point))


Next point using ei: [0.980000, 0.020000, 0.980000, 0.980000]
0.980000-0.020000-0.980000-0.980000


Week-05

In [22]:
from sklearn.gaussian_process.kernels import ConstantKernel, WhiteKernel
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor